In [1]:
from google.colab import userdata
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
print("Key loaded:", PINECONE_API_KEY[:8] + "...")
print("Key length:", len(PINECONE_API_KEY))

Key loaded: pcsk_74e...
Key length: 75


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q pinecone sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 22.6 MB/s eta 0:00:00


In [4]:
import pandas as pd

train_df = pd.read_csv("/content/drive/MyDrive/contract-ai/train.csv")
val_df = pd.read_csv("/content/drive/MyDrive/contract-ai/val.csv")
test_df = pd.read_csv("/content/drive/MyDrive/contract-ai/test.csv")

all_clauses = pd.concat([train_df, val_df, test_df], ignore_index=True)
print(all_clauses.shape)

(13718, 4)


In [5]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(embedder.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384


/tmp/ipykernel_1330/4255289260.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(embedder.get_sentence_embedding_dimension())


In [6]:
clause_texts = all_clauses["clause_text"].tolist()

embeddings = embedder.encode(
    clause_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(embeddings.shape)

Batches:   0%|          | 0/215 [00:00<?, ?it/s]

(13718, 384)


In [7]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "contract-clauses"

if index_name not in [idx["name"] for idx in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print("Index created")
else:
    print("Index already exists")

index = pc.Index(index_name)

Index already exists


In [8]:
print("key length:", len(PINECONE_API_KEY))

key length: 75


In [9]:
all_clauses = all_clauses.reset_index(drop=True)
all_clauses["clause_id"] = ["clause_" + str(i) for i in range(len(all_clauses))]

vectors_to_upload = []
for i, row in all_clauses.iterrows():
    vectors_to_upload.append({
        "id": row["clause_id"],
        "values": embeddings[i].tolist(),
        "metadata": {
            "contract": row["contract"],
            "clause_text": row["clause_text"][:1000],  # Pinecone metadata has size limits
            "label": row["label"],
        }
    })

print(len(vectors_to_upload), "vectors ready")
print(vectors_to_upload[0]["id"], vectors_to_upload[0]["metadata"]["label"])

13718 vectors ready
clause_0 Parties


In [10]:
batch_size = 100

for i in range(0, len(vectors_to_upload), batch_size):
    batch = vectors_to_upload[i:i + batch_size]
    index.upsert(vectors=batch)
    if i % 1000 == 0:
        print(f"Uploaded {i}/{len(vectors_to_upload)}")

print("Upload complete")

Uploaded 0/13718
Uploaded 1000/13718
Uploaded 2000/13718
Uploaded 3000/13718
Uploaded 4000/13718
Uploaded 5000/13718
Uploaded 6000/13718
Uploaded 7000/13718
Uploaded 8000/13718
Uploaded 9000/13718
Uploaded 10000/13718
Uploaded 11000/13718
Uploaded 12000/13718
Uploaded 13000/13718
Upload complete


In [11]:
stats = index.describe_index_stats()
print(stats)

DescribeIndexStatsResponse(dimension=384, total_vector_count=13718, metric='cosine', namespaces=1)


In [12]:
def search_similar_clauses(query_text, top_k=5):
    query_vector = embedder.encode(query_text).tolist()
    results = index.query(vector=query_vector, top_k=top_k, include_metadata=True)

    for match in results["matches"]:
        print(f"Score: {match['score']:.3f} | Label: {match['metadata']['label']}")
        print(match['metadata']['clause_text'][:150])
        print("---")

search_similar_clauses("this agreement can be ended early by either party")

Score: 0.809 | Label: Minimum Commitment
Either Party may terminate this Agreement:
---
Score: 0.789 | Label: Termination For Convenience
Either party may terminate this Agreement at any time on prior written notice to the other.
---
Score: 0.789 | Label: Termination For Convenience
Either party may terminate this Agreement at any time on prior written notice to the other.
---
Score: 0.771 | Label: Post-Termination Services
Upon the termination of this Agreement by either party:
---
Score: 0.759 | Label: Notice Period To Terminate Renewal
Either party may cancel this Agreement effective on the last day of the Initial Term, or
---


In [13]:
search_similar_clauses("the company must keep customer information private")

Score: 0.716 | Label: Audit Rights
Any Records or information accessed or otherwise obtained by Customer or its representatives during any such inspection or audit or any visit at any F
---
Score: 0.691 | Label: Other
(ii) not to disclose or transfer the Information to others without the Company's written permission.
---
Score: 0.651 | Label: Audit Rights
Any Records or information accessed or otherwise obtained by Customer or its representatives in connection with any audit (including any audit pursuan
---
Score: 0.611 | Label: Other
14. Confidential Information. The parties acknowledge that, during the term of this Agreement, each may receive certain Proprietary Information of the
---
Score: 0.600 | Label: Other
12. Confidentiality. All proprietary, technical, experimental, manufacturing, marketing and/or other information disclosed by a party hereto to the ot
---


In [14]:
search_similar_clauses("one party cannot work with competitors")

Score: 0.601 | Label: Non-Compete
Party A may not make any identical or similar cooperation regarding the real estate and home furnishing information, products and data in its real est
---
Score: 0.586 | Label: Non-Compete
Party B shall not advertise, or make any statement favorable for, any competitor having the same or similar business scope as Party A in the services 
---
Score: 0.522 | Label: Parties
each of the Company, on the one hand, and the Marathon Parties, on the other hand, a "Party" to this Agreement, and collectively, the "Parties"
---
Score: 0.494 | Label: Parties
each a "Party" and together the "Parties
---
Score: 0.485 | Label: Parties
Parties
---


In [15]:
search_similar_clauses("payment is due within 30 days")

Score: 0.666 | Label: Liquidated Damages
Payment of Liquidated Damages is due thirty (30) days following termination of this Agreement or on demand.
---
Score: 0.580 | Label: Audit Rights
During the 18-month period following the payment by one party of any amount due under this Agreement to the other party, the party receiving
---
Score: 0.576 | Label: Revenue/Profit Sharing
Such royalty payment shall be payable to Pfizer within thirty (30) days of the end of each Calendar Quarter.
---
Score: 0.574 | Label: Revenue/Profit Sharing
The first Development Fees payment shall be due forty-five days after the end of the Calendar Quarter in which the first Sale of a Development Product
---
Score: 0.559 | Label: Termination For Convenience
Either party may terminate this Agreement at any time with 30 days written notice.
---
